# Building attributes for the study area

Third part of the session. Here I run the `Imageable` library over the imagery of part 1 to get the
footprint, neighborhood, height and material attributes.

The neighborhood attributes need all the buildings of Boston and not only the ones of the study area,
otherwise every building close to the border gets the wrong neighbors. Passing the whole city is very
slow because the library reprojects the whole list on every building, so I pass the ones within 600 m,
which cannot change the result.

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import imageable._utils.image_quality.image_quality_operations as imqual
from imageable import get_building_data_from_gdf
from imageable._extraction.image import ImageCalculator
from imageable._models.huggingface.segformer_segmentation import SegformerSegmentationWrapper

/Users/legariapena.j/Documents/imageable-2026/imageable/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/legariapena.j/Documents/imageable-2026/imageable/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
IMAGEABLE_PATH = Path("/Users/legariapena.j/Documents/imageable-2026/imageable")

QUALITY_DATA_PATH = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/"
"data/2026_08_03_imagery_quality_assessment")

OUTPUT_PATH = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/"
"data/2026_08_04_building_attributes")

PARENT_IMAGES = QUALITY_DATA_PATH/Path("images")
STUDY_AREA_PATH = QUALITY_DATA_PATH/Path("boston_buildings_study_area.geojson")

ALL_BOSTON_PATH = Path("/Users/legariapena.j/Documents/sampling/boston_buildings.geojson")

BATCHES_PATH = OUTPUT_PATH/Path("batches")
BATCHES_PATH.mkdir(parents = True, exist_ok = True)

CONTEXT_PATH = OUTPUT_PATH/Path("boston_buildings_context.geojson")
ATTRIBUTES_PATH = OUTPUT_PATH/Path("building_attributes.geojson")
ATTRIBUTES_TABLE_PATH = OUTPUT_PATH/Path("building_attributes.parquet")

### Footprints of the study area and of the whole city

Two of the footprints come as MultiPolygon and the library only reads polygons, so I keep their largest part.

In [3]:
gdf_study = gpd.read_file(STUDY_AREA_PATH)

gdf_study["building_id"] = ["building_"+str(i) for i in range(0,len(gdf_study))]

geometries = []
for i in range(0,len(gdf_study)):
    geometry = gdf_study.geometry.iloc[i]
    if(geometry.geom_type == "MultiPolygon"):
        geometry = max(geometry.geoms, key = lambda part: part.area)

    geometries.append(geometry)

gdf_study["geometry"] = geometries

gdf_all_boston = gpd.read_file(ALL_BOSTON_PATH)

print(f"Study area buildings: {len(gdf_study)}")
print(f"Boston buildings: {len(gdf_all_boston)}")
print(f"Study area crs: {gdf_study.crs}, Boston crs: {gdf_all_boston.crs}")

study_object_ids = set(gdf_study["OBJECTID"])
all_object_ids = set(gdf_all_boston["OBJECTID"])
print(f"The study area is a subset of the city dataset: {study_object_ids.issubset(all_object_ids)}")

Study area buildings: 2456
Boston buildings: 128564
Study area crs: EPSG:4326, Boston crs: EPSG:4326
The study area is a subset of the city dataset: True


### Buildings that can be neighbors

In [4]:
NEIGHBOR_RADIUS = 600.0
CONTEXT_BUFFER = 1000.0

study_projected = gdf_study.to_crs(3857)
boston_projected = gdf_all_boston.to_crs(3857)

search_area = study_projected.geometry.union_all().buffer(CONTEXT_BUFFER)
context_mask = boston_projected.geometry.intersects(search_area)

gdf_context = gdf_all_boston[context_mask].copy()

print(f"Buildings that can be neighbors: {len(gdf_context)} out of {len(gdf_all_boston)}")

gdf_context.to_file(CONTEXT_PATH, driver = "GeoJSON")
print(f"Saved the context buildings to {CONTEXT_PATH}")

del gdf_all_boston
del boston_projected
del study_projected

Buildings that can be neighbors: 9856 out of 128564


Saved the context buildings to /Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_08_04_building_attributes/boston_buildings_context.geojson


### Facade mask

The library only fills the image attributes when it gets a mask and the public API
never builds one, so I build it here with the same segmentation of part 2.

In [5]:
model_facade = SegformerSegmentationWrapper(
    model_name = "nvidia/segformer-b5-finetuned-ade-640-640",
    device = "mps",
    palette_path = str(IMAGEABLE_PATH/"src"/"imageable"/"assets"/"ade20k_palette.json")
)
model_facade.load_model()

print(f"Facade model loaded: {model_facade.is_loaded()}")

def get_building_mask(img):
    seg_raw = model_facade.predict(img)
    seg_raw = seg_raw.squeeze().astype("uint8") + 1
    seg = model_facade.remap_labels(seg_raw, imqual.FACADE_REMAPPING)

    return seg == imqual.FACADE_LABEL

IMAGE_ATTRIBUTE_NAMES = [
    "average_red_channel_value",
    "average_green_channel_value",
    "average_blue_channel_value",
    "average_brightness",
    "average_vividness",
    "mask_area",
    "mask_length",
    "mask_complexity",
    "number_of_edges",
    "number_of_vertices"
]

def add_image_attributes(batch_result):
    attribute_values = {}
    for attribute_name in IMAGE_ATTRIBUTE_NAMES:
        attribute_values[attribute_name] = []

    for i in range(0,len(batch_result)):
        building_id = batch_result["building_id"].iloc[i]
        image_path = PARENT_IMAGES/Path(building_id)/Path("image.jpg")

        image_attributes = {}
        if(image_path.exists()):
            img = np.array(Image.open(image_path).convert("RGB"))
            building_mask = get_building_mask(img)

            image_calculator = ImageCalculator(img = img, building_mask = building_mask)
            image_attributes = image_calculator.extract_all_features()

        for attribute_name in IMAGE_ATTRIBUTE_NAMES:
            value = image_attributes.get(attribute_name, None)
            attribute_values[attribute_name].append(None if value is None else float(value))

    for attribute_name in IMAGE_ATTRIBUTE_NAMES:
        batch_result[attribute_name] = attribute_values[attribute_name]

    return batch_result

The image processor of type `SegformerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/1172 [00:00<00:00, 9258.95it/s, Materializing param=decode_head.batch_norm.bias]

Loading weights:   0%|          | 1/1172 [00:00<00:02, 526.00it/s, Materializing param=decode_head.batch_norm.bias] 

Loading weights:   0%|          | 2/1172 [00:00<00:01, 752.41it/s, Materializing param=decode_head.batch_norm.num_batches_tracked]

Loading weights:   0%|          | 2/1172 [00:00<00:01, 657.00it/s, Materializing param=decode_head.batch_norm.num_batches_tracked]

... (2,343 outputs left out)

Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 1740.71it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]

Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 1740.71it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]

Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 1864.29it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]

Facade model loaded: True


### Materials as columns

The library sometimes returns the percentages and the areas wrapped in another dictionary instead of the percentages alone, so that case gets unwrapped before spreading them into columns.

In [6]:
def flatten_material_columns(batch_result):
    percentage_dictionaries = list(batch_result["material_percentages"])
    area_dictionaries = list(batch_result["material_areas"])

    for i in range(0,len(percentage_dictionaries)):
        dictionary = percentage_dictionaries[i]
        if(isinstance(dictionary, dict) and isinstance(dictionary.get("percentages"), dict)):
            area_dictionaries[i] = dictionary.get("areas")
            percentage_dictionaries[i] = dictionary["percentages"]

    material_names = []
    for i in range(0,len(percentage_dictionaries)):
        dictionary = percentage_dictionaries[i]
        if(isinstance(dictionary, dict)):
            for material_name in dictionary:
                if(material_name not in material_names):
                    material_names.append(material_name)

    material_names = sorted(material_names)

    for material_name in material_names:
        percentages = []
        areas = []
        for i in range(0,len(percentage_dictionaries)):
            percentage_dictionary = percentage_dictionaries[i]
            area_dictionary = area_dictionaries[i]

            if(isinstance(percentage_dictionary, dict)):
                percentages.append(percentage_dictionary.get(material_name, None))
            else:
                percentages.append(None)

            if(isinstance(area_dictionary, dict)):
                areas.append(area_dictionary.get(material_name, None))
            else:
                areas.append(None)

        batch_result["material_pct_"+material_name] = percentages
        batch_result["material_area_"+material_name] = areas

    return batch_result.drop(columns = ["material_percentages", "material_areas"])

### Extraction

One call per batch, and every batch is saved as soon as it is done so that a crash in
the middle does not throw away the whole run. Batches already on disk are skipped, so running the
notebook again continues where it stopped.

In [7]:
HEIGHT_MODE = "cluster_then_predict"
BATCH_SIZE = 50

n_buildings = len(gdf_study)
batch_starts = list(range(0, n_buildings, BATCH_SIZE))
failed_batches = []

print(f"{len(batch_starts)} batches of at most {BATCH_SIZE} buildings")

for i in tqdm(range(0,len(batch_starts))):
    start = batch_starts[i]
    end = min(start + BATCH_SIZE, n_buildings)
    batch_path = BATCHES_PATH/Path("batch_"+str(start)+"_"+str(end)+".parquet")

    if(batch_path.exists()):
        continue

    batch_gdf = gdf_study.iloc[start:end]

    try:
        batch_result = get_building_data_from_gdf(
            batch_gdf,
            "",
            images_dir = PARENT_IMAGES,
            all_city_buildings_gdf = gdf_context,
            neighbor_radius = NEIGHBOR_RADIUS,
            height_mode = HEIGHT_MODE,
            id_column = "building_id",
            output_format = "gdf",
            verbose = False
        )

        batch_result = add_image_attributes(batch_result)
        batch_result = flatten_material_columns(batch_result)
        batch_result.to_parquet(batch_path)

    except Exception as error:
        print(f"Batch {start}-{end} failed: {error}")
        failed_batches.append((start, end))
        continue

print(f"Failed batches: {failed_batches}")

50 batches of at most 50 buildings


  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:00<00:00, 60107.54it/s]

Failed batches: []


### Put the batches together

In [8]:
batch_paths = sorted(BATCHES_PATH.glob("batch_*.parquet"), key = lambda path: int(path.stem.split("_")[1]))

batch_results = []
for i in range(0,len(batch_paths)):
    batch_results.append(gpd.read_parquet(batch_paths[i]))

gdf_attributes = pd.concat(batch_results, ignore_index = True)
gdf_attributes = gpd.GeoDataFrame(gdf_attributes, geometry = "geometry", crs = gdf_study.crs)

reference_columns = gdf_study[["building_id", "OBJECTID", "BLDG_HGT_2010"]].rename(
    columns = {"OBJECTID": "object_id", "BLDG_HGT_2010": "reference_height"}
)
gdf_attributes = gdf_attributes.merge(reference_columns, on = "building_id", how = "left")

print(f"Extracted attributes for {len(gdf_attributes)} buildings")
print(f"Columns: {len(gdf_attributes.columns)}")

gdf_attributes.to_parquet(ATTRIBUTES_TABLE_PATH)
gdf_attributes.to_file(ATTRIBUTES_PATH, driver = "GeoJSON")

print(f"Saved the attributes to {ATTRIBUTES_PATH}")
print(f"and to {ATTRIBUTES_TABLE_PATH}")

Extracted attributes for 2456 buildings
Columns: 83
Saved the attributes to /Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_08_04_building_attributes/building_attributes.geojson
and to /Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_08_04_building_attributes/building_attributes.parquet


### Did it work?

The heights of the city dataset are in feet: the largest one of the whole city is
788, and 788 feet is 240 m, the height of 200 Clarendon. So they get converted before comparing.

In [9]:
FEET_TO_METERS = 0.3048

gdf_attributes["reference_height_meters"] = FEET_TO_METERS*gdf_attributes["reference_height"]

heights = gdf_attributes["building_height"].to_numpy(dtype = float)
reference_heights = gdf_attributes["reference_height_meters"].to_numpy(dtype = float)

valid_height = (heights > 0) & np.isfinite(reference_heights)

print(f"Buildings with an estimated height: {int((heights > 0).sum())} out of {len(gdf_attributes)}")
print(f"Buildings with a reference height too: {int(valid_height.sum())}")

errors = heights[valid_height] - reference_heights[valid_height]
print(f"Mean absolute error against the city dataset: {np.mean(np.abs(errors)):.2f} m")
print(f"Bias: {np.mean(errors):+.2f} m")
print(f"Correlation: {np.corrcoef(heights[valid_height], reference_heights[valid_height])[0,1]:.3f}")

material_columns = [column for column in gdf_attributes.columns if column.startswith("material_pct_")]
print(f"\nMaterial columns: {len(material_columns)}")
print(f"Buildings with materials: {int(gdf_attributes[material_columns].notna().any(axis = 1).sum())}")
print(f"Buildings with image attributes: {int((gdf_attributes['mask_area'] > 0).sum())}")
print(f"Buildings with neighbors: {int((gdf_attributes['neighbor_count'] > 0).sum())}")

Buildings with an estimated height: 2053 out of 2456
Buildings with a reference height too: 1944
Mean absolute error against the city dataset: 6.08 m
Bias: -0.16 m
Correlation: 0.225

Material columns: 20
Buildings with materials: 2447
Buildings with image attributes: 2369
Buildings with neighbors: 2456


In [10]:
empty_columns = []
for column in gdf_attributes.columns:
    if(column in ("geometry", "building_id")):
        continue

    values = pd.to_numeric(gdf_attributes[column], errors = "coerce")
    if(values.notna().sum() > 0 and (values.fillna(0) == 0).all()):
        empty_columns.append(column)

print("Columns that came out empty:")
for i in range(0,len(empty_columns)):
    print(f"  {empty_columns[i]}")

Columns that came out empty:
  average_window_x
  average_window_y
  average_door_x
  average_door_y
  number_of_windows
  number_of_doors
  material_pct_water
  material_area_water
